In [1]:
import numpy as np
import matplotlib.pyplot as plt
from FastBEMT.JobParameters import LowFidelityParameters
from FastBEMT.Propeller import Propeller
import scienceplots
plt.style.use(['science','no-latex'])
from FastBEMT.DataLoader import *

blade_dict = load_propeller_dict('Pareto3')

params = LowFidelityParameters(
    rpm=7000,
    a_inf=343,
    rho=1.225,
    mu=1.81e-5,
    n_blades=2,
    p_ref=2e-5,
    revolutions=5,
    num_obs_times_per_rev=100,
    device='cuda',
)

In [2]:
propeller = Propeller(
    propeller_geometry=blade_dict,
    params=params,
)
v_inf = 0
propeller.run_bemt(v_inf=v_inf)

In [3]:
from FastBEMT.Stress import BladeStressCalculator

stress_calc = BladeStressCalculator(propeller=propeller)
sigma_c, sigma_b = stress_calc.blade_stress_report(material_rho=2700, show = False)

In [4]:
import plotly.graph_objects as go

geom = propeller.geometry
r = np.asarray(geom["r"])
chord = np.asarray(geom["chord"])
twist = np.radians(np.asarray(geom["twist"]))
airfoils = geom["airfoil"]

sigma_c = np.asarray(sigma_c)
sigma_b = np.asarray(sigma_b)

if sigma_b.ndim == 1:
    sigma_total = sigma_c + sigma_b
    sigma_total = sigma_total[:, np.newaxis]
else:
    sigma_total = sigma_b + sigma_c[:, np.newaxis]

n_sections = len(r)
n_points = airfoils[0].shape[0]

X = np.zeros((n_sections, n_points))
Y = np.zeros((n_sections, n_points))
Z = np.zeros((n_sections, n_points))
S = np.zeros((n_sections, n_points))

for i in range(n_sections):
    coords = np.asarray(airfoils[i])
    x_local = coords[:, 0] * chord[i]
    z_local = coords[:, 1] * chord[i]

    if hasattr(propeller, "com_shift_forward") and hasattr(propeller, "com_shift_up"):
        x_local = x_local + propeller.com_shift_forward[i] * chord[i]
        z_local = z_local + propeller.com_shift_up[i] * chord[i]

    cos_t = np.cos(twist[i])
    sin_t = np.sin(twist[i])
    x_rot = x_local * cos_t + z_local * sin_t
    z_rot = -x_local * sin_t + z_local * cos_t

    X[i, :] = x_rot
    Y[i, :] = r[i]
    Z[i, :] = z_rot

    if sigma_total.ndim == 2 and sigma_total.shape[1] == n_points:
        S[i, :] = sigma_total[i, :]
    else:
        S[i, :] = sigma_total[i]

S_mpa = S / 1e6

fig = go.Figure(
    data=go.Surface(
        x=X,
        y=Y,
        z=Z,
        surfacecolor=S_mpa,
        colorscale="Viridis",
        colorbar=dict(title="Stress [MPa]"),
    )
 )

fig.update_layout(
    title="Blade Stress Distribution",
    scene=dict(
        xaxis_title="Chordwise x [m]",
        yaxis_title="Radius y [m]",
        zaxis_title="Thickness z [m]",
        aspectmode="data",
    ),
    margin=dict(l=0, r=0, t=40, b=0),
 )

fig.show()